# 🏠 Irish Rental Crisis — Analysis & Prediction Platform
## Notebook 02: Data Cleaning & Combining

---

### What This Notebook Does
This notebook takes the eight raw datasets loaded in Notebook 01 and prepares them for analysis.

The work here is split into two parts:
- **Part 1 — Cleaning:** Fix column names, drop redundant columns, handle nulls, filter to the statistics we actually need
- **Part 2 — Combining:** Merge the supporting datasets into two context files — one at county level, one at LEA level

No analysis or visualisation happens here. This notebook answers one question: **"Is the data clean, consistent, and ready to explore?"**

---

### Inputs
| File | Description |
|------|-------------|
| `RIQ02_rent_index_quarterly.csv` | RTB quarterly rent index (2015–2025) |
| `RIA02_rent_index_annual.csv` | RTB annual rent index (2008–2024) |
| `HAP10_hap_properties.csv` | HAP properties by Local Electoral Area (2015–2022) |
| `TRS17_landlord_income.csv` | Median landlord income by Local Authority (2019) |
| `TRS18_landlord_income_lea.csv` | Median landlord income by Local Electoral Area (2019) |
| `TRS21_rent_affordability.csv` | Rent as % of disposable income by Local Authority (2019) |
| `TRS22_rent_affordability_lea.csv` | Rent as % of disposable income by Local Electoral Area (2019) |
| `FY001_population_by_county.csv` | Population by county — Census 2011, 2016, 2022 |

### Outputs
| File | Rows | Description |
|------|------|-------------|
| `rent_quarterly_clean.csv` | 210,865 | Cleaned RIQ02 — core quarterly rent data |
| `rent_annual_clean.csv` | 109,380 | Cleaned RIA02 — annual rent back to 2008 |
| `context_county_clean.csv` | 29 | Landlord income + affordability + population at county level |
| `context_lea_clean.csv` | 1,328 | HAP + landlord income + affordability at LEA level |

---

### Key Decisions Made in This Notebook
| Decision | Reason |
|----------|--------|
| Dropped structural nulls from RIQ02 and RIA02 | RTB only reports rent where enough tenancies exist — nulls are expected, not errors |
| Filled HAP10 nulls with 0 | HAP scheme had not reached those LEAs in 2015/2016 — programme rollout gap |
| Kept only Median Total Gross Income from TRS17/18 | The other statistic (rental income alone) is less useful without gross income context |
| Kept Median Rent % and 30% threshold from TRS21/22 | These two directly measure affordability — 35% and 40% thresholds are redundant |
| Averaged Cork and Galway City/County entries | CSO reports them separately but FY001 treats them as one — averaged to align granularity |
| Split Dublin population equally across 4 sub-councils | FY001 only has one Dublin entry — equal split is the fairest approximation |
| Used 2022 census population only | Most recent census, closest to our rent data timeframe |

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

print('Libraries imported.')

Libraries imported.


## 2. Load Raw Datasets

In [2]:
# Load all 8 raw datasets
df_riq   = pd.read_csv('/content/RIQ02_rent_index_quarterly.csv', on_bad_lines='skip', engine='python')
df_ria   = pd.read_csv('/content/RIA02_rent_index_annual.csv')
df_hap   = pd.read_csv('/content/HAP10_hap_properties.csv')
df_trs17 = pd.read_csv('/content/TRS17_landlord_income.csv')
df_trs18 = pd.read_csv('/content/TRS18_landlord_income_lea.csv')
df_trs21 = pd.read_csv('/content/TRS21_rent_affordability.csv')
df_trs22 = pd.read_csv('/content/TRS22_rent_affordability_lea.csv')
df_pop   = pd.read_csv('/content/FY001_population_by_county.csv')

print('All 8 datasets loaded.')

All 8 datasets loaded.


## 3. Clean RIQ02 — Quarterly Rent Index

**Source:** CSO PxStat — RIQ02
**Coverage:** 2015 Q1 to 2025 Q2

RIQ02 is the core dataset for the entire project. It has 786,744 rows in the raw file but only ~210,000 have an actual rent value. The rest are structural nulls — the RTB only publishes a figure where a statistically sufficient number of tenancies exist. These nulls are expected and are dropped here.

We also extract Year and Quarter number from the Quarter string (e.g. "2019Q3" → Year=2019, Q_Num=3) to make time-based analysis easier later.

In [3]:
# Drop single-value columns that add no information
df_riq = df_riq.drop(columns=['STATISTIC Label', 'UNIT'])

# Rename VALUE to something cleaner
df_riq = df_riq.rename(columns={'VALUE': 'Value'})

# Drop structural nulls — rows where RTB did not report a rent figure
df_riq = df_riq.dropna(subset=['Value']).copy()

# Pull Year and Quarter number out of the Quarter string e.g. "2019Q3"
df_riq['Year']  = df_riq['Quarter'].str.extract(r'(\d{4})').astype(int)
df_riq['Q_Num'] = df_riq['Quarter'].str.extract(r'Q(\d)').astype(int)

# Tidy up column names
df_riq = df_riq.rename(columns={
    'Number of Bedrooms' : 'Bedrooms',
    'Property Type'      : 'PropertyType'
})

print(f'Rows after cleaning : {df_riq.shape[0]:,}')
print(f'Columns             : {list(df_riq.columns)}')
print(f'Nulls               : {df_riq.isnull().sum().sum()}')
df_riq.head()

Rows after cleaning : 210,865
Columns             : ['Quarter', 'Bedrooms', 'PropertyType', 'Location', 'Value', 'Year', 'Q_Num']
Nulls               : 0


,Quarter,Bedrooms,PropertyType,Location,Value,Year,Q_Num
0,2015Q1,All bedrooms,All property types,Carlow,595.48,2015,1
1,2015Q1,All bedrooms,All property types,Carlow Town,621.30,2015,1
2,2015Q1,All bedrooms,All property types,"Graiguecullen, Carlow",546.54,2015,1
3,2015Q1,All bedrooms,All property types,"Tullow, Carlow",569.47,2015,1
4,2015Q1,All bedrooms,All property types,Cavan,456.46,2015,1


## 4. Clean RIA02 — Annual Rent Index

**Source:** CSO PxStat — RIA02
**Coverage:** 2008 to 2024

RIA02 has the same structure as RIQ02 but annual instead of quarterly. It extends the time series back to 2008, giving us the full picture of the rental crisis from before it started.

Same cleaning steps apply — drop redundant columns, drop structural nulls, standardise column names.

In [4]:
# Drop single-value columns
df_ria = df_ria.drop(columns=['STATISTIC Label', 'UNIT'])

# Rename VALUE
df_ria = df_ria.rename(columns={'VALUE': 'Value'})

# Drop structural nulls
df_ria = df_ria.dropna(subset=['Value']).copy()

# Tidy column names
df_ria = df_ria.rename(columns={
    'Number of Bedrooms' : 'Bedrooms',
    'Property Type'      : 'PropertyType'
})

print(f'Rows after cleaning : {df_ria.shape[0]:,}')
print(f'Columns             : {list(df_ria.columns)}')
print(f'Nulls               : {df_ria.isnull().sum().sum()}')
df_ria.head()

Rows after cleaning : 109,380
Columns             : ['Year', 'Bedrooms', 'PropertyType', 'Location', 'Value']
Nulls               : 0


,Year,Bedrooms,PropertyType,Location,Value
0,2008,All bedrooms,All property types,Carlow,748.48
1,2008,All bedrooms,All property types,Carlow Town,811.53
2,2008,All bedrooms,All property types,"Graiguecullen, Carlow",711.35
3,2008,All bedrooms,All property types,"Tullow, Carlow",720.04
4,2008,All bedrooms,All property types,Cavan,571.72


## 5. Clean HAP10 — Housing Assistance Payment Properties

**Source:** CSO PxStat — HAP10
**Coverage:** 2015 to 2022, Local Electoral Area level

HAP10 tracks how many properties are registered under the Housing Assistance Payment scheme in each LEA each year. High HAP concentration in an area means high social housing pressure.

**Null handling:** 47 nulls exist in 2015 and 2016 only. HAP launched in 2014 and rolled out gradually — these LEAs had no registered HAP properties in those early years. Decision: fill with 0.

The LEA column contains both area name and county in the format "Area, County" — we split this into two separate columns.

In [5]:
# Drop single-value columns
df_hap = df_hap.drop(columns=['STATISTIC Label', 'UNIT'])

# Rename VALUE
df_hap = df_hap.rename(columns={
    'VALUE'                : 'HAP_Properties',
    'Local Electoral Area' : 'LEA'
})

# Fill nulls with 0 — HAP had not reached these LEAs in 2015/2016
df_hap['HAP_Properties'] = df_hap['HAP_Properties'].fillna(0)

# Split LEA into Area and County — format is "Area, County"
df_hap['Area']   = df_hap['LEA'].str.split(',').str[0].str.strip()
df_hap['County'] = df_hap['LEA'].str.split(',').str[1].str.strip()

print(f'Rows   : {df_hap.shape[0]:,}')
print(f'Columns: {list(df_hap.columns)}')
print(f'Nulls  : {df_hap.isnull().sum().sum()}')
print(f'Unique Counties: {df_hap["County"].nunique()}')
df_hap.head()

Rows   : 1,328
Columns: ['Year', 'LEA', 'HAP_Properties', 'Area', 'County']
Nulls  : 0
Unique Counties: 31


,Year,LEA,HAP_Properties,Area,County
0,2015,"Borris-In-Ossory-Mountmellick, Laois",0.0,Borris-In-Ossory-Mountmellick,Laois
1,2015,"Portlaoise, Laois",0.0,Portlaoise,Laois
2,2015,"Graiguecullen -Portarlington, Laois",0.0,Graiguecullen -Portarlington,Laois
3,2015,"Carlow, Carlow",48.0,Carlow,Carlow
4,2015,"Tullow, Carlow",18.0,Tullow,Carlow


## 6. Clean TRS17 and TRS21 — County Level Income & Affordability

**Source:** CSO PxStat — TRS17, TRS21
**Coverage:** 2019 only, Local Authority level

TRS17 and TRS21 each contain multiple statistics per county. We only need one from each:
- **TRS17:** Median Total Gross Income of RTB Landlords — tells us the income profile of landlords by county
- **TRS21:** Median Individual Tenant Rent as a % of Disposable Income — direct measure of affordability

We also keep the 30% affordability threshold from TRS21 as a second indicator.

**County name cleaning:** The raw data has names like "Cork City Council" and "Cork County Council". We strip the council suffix to get clean county names. Cork and Galway appear as both city and county entries — we average them to get one value per county.

In [6]:
# Reload raw files so we can filter by Statistic Label
df_trs17_raw = pd.read_csv('/content/TRS17_landlord_income.csv')
df_trs21_raw = pd.read_csv('/content/TRS21_rent_affordability.csv')

# Keep only the statistic we need from TRS17
df_trs17 = df_trs17_raw[df_trs17_raw['Statistic Label'] == 'Median Total Gross Income of RTB Landlords'].copy()
df_trs17 = df_trs17.drop(columns=['Statistic Label', 'UNIT'])
df_trs17 = df_trs17.rename(columns={
    'VALUE'           : 'Median_Gross_Income',
    'Local Authority' : 'County',
    'Rental Year'     : 'Year'
})

# Keep two statistics from TRS21
df_trs21_median = df_trs21_raw[df_trs21_raw['Statistic Label'] == 'Median Individual Tenant Rent as a % of Disposable Income'].copy()
df_trs21_30pct  = df_trs21_raw[df_trs21_raw['Statistic Label'] == 'Percentage of Tenants who Pay 30% or More of Their Disposable Income on Rent'].copy()

for df_temp in [df_trs21_median, df_trs21_30pct]:
    df_temp.drop(columns=['Statistic Label', 'UNIT'], inplace=True)
    df_temp.rename(columns={'Local Authority': 'County', 'Rental Year': 'Year'}, inplace=True)

df_trs21_median = df_trs21_median.rename(columns={'VALUE': 'Rent_Pct_Disposable_Income'})
df_trs21_30pct  = df_trs21_30pct.rename(columns={'VALUE': 'Pct_Tenants_Over30pct'})

# Clean county names — strip council suffixes
for df_temp in [df_trs17, df_trs21_median, df_trs21_30pct]:
    df_temp['County'] = df_temp['County'].str.replace(' County Council', '', regex=False)
    df_temp['County'] = df_temp['County'].str.replace(' City Council',   '', regex=False)
    # Fix truncated names from the replace above
    df_temp['County'] = df_temp['County'].str.replace('Limerick City and', 'Limerick', regex=False)
    df_temp['County'] = df_temp['County'].str.replace('Waterford City and', 'Waterford', regex=False)

# Cork and Galway appear as both city and county — average to get one value per county
df_trs17        = df_trs17.groupby(['Year', 'County'])['Median_Gross_Income'].mean().reset_index()
df_trs21_median = df_trs21_median.groupby(['Year', 'County'])['Rent_Pct_Disposable_Income'].mean().reset_index()
df_trs21_30pct  = df_trs21_30pct.groupby(['Year', 'County'])['Pct_Tenants_Over30pct'].mean().reset_index()

print(f'TRS17        : {df_trs17.shape[0]} rows | {list(df_trs17.columns)}')
print(f'TRS21 median : {df_trs21_median.shape[0]} rows | {list(df_trs21_median.columns)}')
print(f'TRS21 30pct  : {df_trs21_30pct.shape[0]} rows | {list(df_trs21_30pct.columns)}')

TRS17        : 29 rows | ['Year', 'County', 'Median_Gross_Income']
TRS21 median : 29 rows | ['Year', 'County', 'Rent_Pct_Disposable_Income']
TRS21 30pct  : 29 rows | ['Year', 'County', 'Pct_Tenants_Over30pct']


## 7. Clean TRS18 and TRS22 — LEA Level Income & Affordability

**Source:** CSO PxStat — TRS18, TRS22
**Coverage:** 2019 only, Local Electoral Area level

Same datasets as TRS17/21 but at finer LEA granularity. Same filtering logic applies — keep one statistic from TRS18 and two from TRS22.

LEA column is split into Area and County the same way as HAP10.

In [7]:
# Reload raw files
df_trs18_raw = pd.read_csv('/content/TRS18_landlord_income_lea.csv')
df_trs22_raw = pd.read_csv('/content/TRS22_rent_affordability_lea.csv')

# Filter TRS18 — keep Median Total Gross Income only
df_trs18 = df_trs18_raw[df_trs18_raw['Statistic Label'] == 'Median Total Gross Income of RTB Landlords'].copy()
df_trs18 = df_trs18.drop(columns=['Statistic Label', 'UNIT'])
df_trs18 = df_trs18.rename(columns={
    'VALUE'                : 'Median_Gross_Income',
    'Local Electoral Area' : 'LEA',
    'Rental Year'          : 'Year'
})
df_trs18['Area']   = df_trs18['LEA'].str.split(',').str[0].str.strip()
df_trs18['County'] = df_trs18['LEA'].str.split(',').str[1].str.strip()

# Filter TRS22 — keep median affordability and 30% threshold
df_trs22_median = df_trs22_raw[df_trs22_raw['Statistic Label'] == 'Median Individual Tenant Rent as a % of Disposable Income'].copy()
df_trs22_30pct  = df_trs22_raw[df_trs22_raw['Statistic Label'] == 'Percentage of Tenants who Pay 30% or More of Their Disposable Income on Rent'].copy()

for df_temp, col_name in [(df_trs22_median, 'Rent_Pct_Disposable_Income'), (df_trs22_30pct, 'Pct_Tenants_Over30pct')]:
    df_temp.drop(columns=['Statistic Label', 'UNIT'], inplace=True)
    df_temp.rename(columns={'VALUE': col_name, 'Local Electoral Area': 'LEA', 'Rental Year': 'Year'}, inplace=True)
    df_temp['Area']   = df_temp['LEA'].str.split(',').str[0].str.strip()
    df_temp['County'] = df_temp['LEA'].str.split(',').str[1].str.strip()

print(f'TRS18        : {df_trs18.shape[0]} rows | {list(df_trs18.columns)}')
print(f'TRS22 median : {df_trs22_median.shape[0]} rows | {list(df_trs22_median.columns)}')
print(f'TRS22 30pct  : {df_trs22_30pct.shape[0]} rows | {list(df_trs22_30pct.columns)}')

TRS18        : 166 rows | ['Year', 'LEA', 'Median_Gross_Income', 'Area', 'County']
TRS22 median : 166 rows | ['Year', 'LEA', 'Rent_Pct_Disposable_Income', 'Area', 'County']
TRS22 30pct  : 166 rows | ['Year', 'LEA', 'Pct_Tenants_Over30pct', 'Area', 'County']


## 8. Clean FY001 — Population by County

**Source:** CSO PxStat — FY001
**Coverage:** Census years 2011, 2016, 2022

FY001 has population by county across three census years, broken down by sex. We only need the total (both sexes) and only at county level — we drop the national State aggregate row.

In [8]:
# Keep both sexes total only
df_pop = df_pop[df_pop['Sex'] == 'Both sexes'].copy()

# Drop the national total row — we need county level only
df_pop = df_pop[df_pop['County'] != 'State'].copy()

# Drop columns we don't need
df_pop = df_pop.drop(columns=['Statistic Label', 'UNIT', 'Sex'])

# Rename VALUE
df_pop = df_pop.rename(columns={'VALUE': 'Population'})

print(f'Rows   : {df_pop.shape[0]}')
print(f'Columns: {list(df_pop.columns)}')
print(f'Counties: {df_pop["County"].nunique()} (expected 26)')
df_pop.head()

Rows   : 78
Columns: ['CensusYear', 'County', 'Population']
Counties: 26 (expected 26)


,CensusYear,County,Population
3,2011,Carlow,54612
6,2011,Dublin,1273069
9,2011,Kildare,210312
12,2011,Kilkenny,95419
15,2011,Laois,80559


## 9. Cleaning Summary

All 8 datasets cleaned. Quick check to confirm zero nulls across everything before we start combining.

In [9]:
print('=' * 60)
print('CLEANING SUMMARY')
print('=' * 60)

datasets = {
    'RIQ02 — Quarterly Rent'        : df_riq,
    'RIA02 — Annual Rent'           : df_ria,
    'HAP10 — HAP Properties'        : df_hap,
    'TRS17 — Landlord Income (LA)'  : df_trs17,
    'TRS18 — Landlord Income (LEA)' : df_trs18,
    'TRS21 — Affordability (LA)'    : df_trs21_median,
    'TRS22 — Affordability (LEA)'   : df_trs22_median,
    'FY001 — Population'            : df_pop,
}

for name, dataset in datasets.items():
    nulls = dataset.isnull().sum().sum()
    print(f'{name:<35} {dataset.shape[0]:>8,} rows  |  {dataset.shape[1]} cols  |  {nulls} nulls')

CLEANING SUMMARY
RIQ02 — Quarterly Rent               210,865 rows  |  7 cols  |  0 nulls
RIA02 — Annual Rent                  109,380 rows  |  5 cols  |  0 nulls
HAP10 — HAP Properties                 1,328 rows  |  5 cols  |  0 nulls
TRS17 — Landlord Income (LA)              29 rows  |  3 cols  |  0 nulls
TRS18 — Landlord Income (LEA)            166 rows  |  5 cols  |  0 nulls
TRS21 — Affordability (LA)                29 rows  |  3 cols  |  0 nulls
TRS22 — Affordability (LEA)              166 rows  |  5 cols  |  0 nulls
FY001 — Population                        78 rows  |  3 cols  |  0 nulls


## 10. Build Context Dataset — County Level

We now combine TRS17, TRS21, and FY001 into a single county-level context dataset.

**Join logic:**
- TRS17 + TRS21 median + TRS21 30pct → joined on County
- Population from FY001 (2022 only) → joined on County

**Dublin issue:** FY001 has one Dublin entry but TRS17/21 have four Dublin sub-councils (Dublin City, Dún Laoghaire-Rathdown, Fingal, South Dublin). We split the Dublin population equally across the four entries as the fairest approximation.

In [10]:
# Use 2022 population only — most recent, closest to our rent data timeframe
df_pop_2022 = df_pop[df_pop['CensusYear'] == 2022][['County', 'Population']].copy()

# Join the three county-level datasets together
df_county_context = df_trs17.merge(df_trs21_median[['County', 'Rent_Pct_Disposable_Income']], on='County', how='left')
df_county_context = df_county_context.merge(df_trs21_30pct[['County', 'Pct_Tenants_Over30pct']], on='County', how='left')
df_county_context = df_county_context.merge(df_pop_2022, on='County', how='left')

# Split Dublin population equally across its 4 sub-councils
dublin_pop    = df_pop_2022[df_pop_2022['County'] == 'Dublin']['Population'].values[0]
dublin_share  = dublin_pop / 4
dublin_counties = ['Dublin', 'Dun Laoghaire-Rathdown', 'Fingal', 'South Dublin']
df_county_context.loc[df_county_context['County'].isin(dublin_counties), 'Population'] = dublin_share

print(f'Shape  : {df_county_context.shape}')
print(f'Columns: {list(df_county_context.columns)}')
print(f'Nulls  : {df_county_context.isnull().sum().sum()}')
df_county_context.head(10)

Shape  : (29, 6)
Columns: ['Year', 'County', 'Median_Gross_Income', 'Rent_Pct_Disposable_Income', 'Pct_Tenants_Over30pct', 'Population']
Nulls  : 0


,Year,County,Median_Gross_Income,Rent_Pct_Disposable_Income,Pct_Tenants_Over30pct,Population
0,2019,Carlow,46865.0,22.00,31.40,61968.0
1,2019,Cavan,43437.0,16.80,22.00,81704.0
2,2019,Clare,46078.0,17.60,21.60,127938.0
3,2019,Cork,53993.5,23.10,33.35,584156.0
4,2019,Donegal,37709.0,18.60,23.90,167084.0
5,2019,Dublin,63321.0,29.30,48.10,364538.5
6,2019,Dun Laoghaire-Rathdown,66256.0,28.50,46.50,364538.5
7,2019,Fingal,58965.0,27.80,43.90,364538.5
8,2019,Galway,53309.5,22.25,32.55,277737.0
9,2019,Kerry,44756.0,20.30,30.00,156458.0


## 11. Build Context Dataset — LEA Level

Combine HAP10, TRS18, and TRS22 into a single LEA-level context dataset.

All three datasets share the same LEA granularity so the join is straightforward.

In [11]:
# Join HAP10 + TRS18 + TRS22 on LEA
df_lea_context = df_hap.merge(
    df_trs18[['LEA', 'Median_Gross_Income', 'Area', 'County']],
    on='LEA', how='left'
)
df_lea_context = df_lea_context.merge(
    df_trs22_median[['LEA', 'Rent_Pct_Disposable_Income']],
    on='LEA', how='left'
)
df_lea_context = df_lea_context.merge(
    df_trs22_30pct[['LEA', 'Pct_Tenants_Over30pct']],
    on='LEA', how='left'
)

# Clean up duplicate Area and County columns from the merge
df_lea_context = df_lea_context.drop(columns=['Area_y', 'County_y'])
df_lea_context = df_lea_context.rename(columns={'Area_x': 'Area', 'County_x': 'County'})

print(f'Shape  : {df_lea_context.shape}')
print(f'Columns: {list(df_lea_context.columns)}')
print(f'Nulls  : {df_lea_context.isnull().sum().sum()}')
df_lea_context.head()

Shape  : (1328, 8)
Columns: ['Year', 'LEA', 'HAP_Properties', 'Area', 'County', 'Median_Gross_Income', 'Rent_Pct_Disposable_Income', 'Pct_Tenants_Over30pct']
Nulls  : 0


,Year,LEA,HAP_Properties,Area,County,Median_Gross_Income,Rent_Pct_Disposable_Income,Pct_Tenants_Over30pct
0,2015,"Borris-In-Ossory-Mountmellick, Laois",0.0,Borris-In-Ossory-Mountmellick,Laois,45467,21.0,31.5
1,2015,"Portlaoise, Laois",0.0,Portlaoise,Laois,49474,21.6,31.4
2,2015,"Graiguecullen -Portarlington, Laois",0.0,Graiguecullen -Portarlington,Laois,51499,20.3,26.6
3,2015,"Carlow, Carlow",48.0,Carlow,Carlow,47541,24.0,35.9
4,2015,"Tullow, Carlow",18.0,Tullow,Carlow,46300,17.5,22.3


## 12. Export Clean Datasets

Export all 4 clean datasets to `/content/` ready for Notebook 03 EDA.

In [12]:
# Export all 4 clean datasets
df_riq.to_csv('/content/rent_quarterly_clean.csv', index=False)
df_ria.to_csv('/content/rent_annual_clean.csv', index=False)
df_county_context.to_csv('/content/context_county_clean.csv', index=False)
df_lea_context.to_csv('/content/context_lea_clean.csv', index=False)

print('Exports complete:')
print(f'  rent_quarterly_clean.csv  — {df_riq.shape[0]:,} rows  | {df_riq.shape[1]} cols')
print(f'  rent_annual_clean.csv     — {df_ria.shape[0]:,} rows  | {df_ria.shape[1]} cols')
print(f'  context_county_clean.csv  — {df_county_context.shape[0]:>6} rows  | {df_county_context.shape[1]} cols')
print(f'  context_lea_clean.csv     — {df_lea_context.shape[0]:,} rows  | {df_lea_context.shape[1]} cols')

Exports complete:
  rent_quarterly_clean.csv  — 210,865 rows  | 7 cols
  rent_annual_clean.csv     — 109,380 rows  | 5 cols
  context_county_clean.csv  —     29 rows  | 6 cols
  context_lea_clean.csv     — 1,328 rows  | 8 cols


## 13. Notebook Summary

### What Was Done
All 8 raw datasets were cleaned, standardised, and combined into 4 analysis-ready files.

### Dataset Overview
| File | Rows | Key Notes |
|------|------|-----------|
| `rent_quarterly_clean.csv` | 210,865 | Structural nulls dropped — only rows with actual rent figures kept |
| `rent_annual_clean.csv` | 109,380 | Annual view back to 2008 — full crisis timeline |
| `context_county_clean.csv` | 29 | County level — landlord income, affordability, population |
| `context_lea_clean.csv` | 1,328 | LEA level — HAP properties, landlord income, affordability |

### Key Observations
- **RIQ02 went from 786,744 rows to 210,865** after dropping structural nulls — 73% of the raw file was empty combinations
- **HAP10 nulls confirmed as rollout gap** — all 47 nulls were in 2015/2016 only, filled with 0
- **Cork and Galway averaged** — CSO reports city and county separately but population data treats them as one
- **Dublin population split equally** across 4 sub-councils as FY001 only has one Dublin entry
- **All 4 output files have zero nulls**

### Next Step
Proceed to **Notebook 03: EDA & Statistical Analysis** where we explore rent trends, affordability patterns, and county-level comparisons using these clean datasets.

In [13]:
print('=' * 55)
print('NOTEBOOK 02 — DATA CLEANING & COMBINING COMPLETE')
print('=' * 55)
print()
print('4 clean datasets ready for EDA:')
print('  rent_quarterly_clean.csv  — 210,865 rows')
print('  rent_annual_clean.csv     — 109,380 rows')
print('  context_county_clean.csv  —      29 rows')
print('  context_lea_clean.csv     —   1,328 rows')
print()
print('Next: Notebook 03 — EDA & Statistical Analysis')

NOTEBOOK 02 — DATA CLEANING & COMBINING COMPLETE

4 clean datasets ready for EDA:
  rent_quarterly_clean.csv  — 210,865 rows
  rent_annual_clean.csv     — 109,380 rows
  context_county_clean.csv  —      29 rows
  context_lea_clean.csv     —   1,328 rows

Next: Notebook 03 — EDA & Statistical Analysis
